# Алгоритм муравьиной колонии (ACO)

In [ ]:
import numpy as np
import osmnx as ox
import pandas as pd
import geopandas as gpd
import networkx as nx

import matplotlib.pyplot as plt

from graphs.algorithms import generate_grid_graph
from shapely.geometry import LineString, Polygon

## Тестовый граф

In [ ]:
G = generate_grid_graph(n=100, m=100, step=100.0, speed=20, diagonals=True)
nx.set_edge_attributes(G, 'undefined', 'highway')

### Проверка алгоритмом HillClimber

In [ ]:
from genesis.states import FirstArrivalUnitState
from genesis.metrics import ArrivalTime, CoverIndex
from genesis.best_points import BestNodeHillClimbing
from genesis.utils import timing

state = FirstArrivalUnitState()
metric = ArrivalTime()

hill_climber = BestNodeHillClimbing(
    state_function  = state,
    metric_function = metric,
    )

best_nodes, best_metric = timing(hill_climber)(env=G)
print('Результат:', best_nodes, best_metric)

# Оценка результата
times, _ = FirstArrivalUnitState()(G, {best_nodes: 'full'})

g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
g_nodes['arrival_time'] = times

In [ ]:

def print_results(g_nodes, units, figsize=(10,10)):
    # Печать карты
    fig, ax = plt.subplots(1,1, figsize=figsize)
    g_nodes.query('arrival_time<=10').plot(ax=ax, color='g', label='Тпр <= 10', marker='s', legend=True)
    g_nodes.query('arrival_time<=20 and arrival_time>10').plot(ax=ax, color='y', label='Тпр > 10', marker='s', legend=True)
    g_nodes.query('arrival_time>20').plot(ax=ax, color='r', label='Тпр > 20', marker='s', legend=True)
    g_nodes.loc[units.keys()].plot(ax=ax, color='b', label='ПСЧ', marker='*', legend=True)
    ax.axis('off')
    ax.legend()
    plt.show()

    # Печать метрик
    print('     Среднее время прибытия: ', round(ArrivalTime()(times), 1), 'мин.')
    print('Максимальное время прибытия: ', round(ArrivalTime(np.max)(times), 1), 'мин.')
    print('                      ИП-10: ', CoverIndex(10)(g_nodes['arrival_time']), '%')
    print('                      ИП-20: ', CoverIndex(20)(g_nodes['arrival_time']), '%')

print_results(g_nodes, {best_nodes: 'full'}, figsize=(10,10))

## Муравьиный алгоритм

### Исходная реализация

In [ ]:
# Для n туров

    # Генерируем m маршрутов муравьев (k посещаемых точек)
        # Случайно выбираем k точек (вероятность выбора зависит от количества феромона)

    # Для каждого маршрута оцениваем метрику
    # исходя из этого оцениваем сколько феромона оставить
    # добавляем феромон
    # испаряем  феромон

In [ ]:
# Реализация ACO в отдельной ячейке.

def aco_best_points(env,
                    state_function,
                    metric_function,
                    num_points=1,
                    num_ants=20,
                    num_iterations=30,
                    alpha=1.0,
                    evaporation_rate=0.1,
                    q=1.0,
                    points_pool=None,
                    random_state=None,
                    **kwargs):
    """Ищет лучшие узлы графа методом муравьиной колонии.

    Args:
        env (nx.Graph): граф.
        state_function (StateBase): вычисление времен прибытия от множества точек.
        metric_function (MetricBase): метрика для оценки качества размещения.
        num_points (int): число точек размещения.
        num_ants (int): число маршрутов (муравьев) за итерацию.
        num_iterations (int): число итераций.
        alpha (float): степень влияния феромона на выбор узлов.
        evaporation_rate (float): скорость испарения феромона.
        q (float): множитель для добавления феромона.
        points_pool (iterable): список узлов кандидатов.
        random_state (int): сид для генератора случайных чисел.
        **kwargs: дополнительные параметры для state_function (например area).

    Returns:
        tuple: (best_nodes, best_metric, pheromone)
    """

    if points_pool is None:
        points_pool = list(env.nodes())
    else:
        points_pool = list(points_pool)

    if len(points_pool) == 0:
        raise ValueError('points_pool не может быть пустым')

    num_points = min(num_points, len(points_pool))
    node_to_index = {node: i for i, node in enumerate(points_pool)}
    pheromone = np.ones(len(points_pool), dtype=float)
    rng = np.random.default_rng(random_state)
    eps = 1e-8

    best_solution = None
    best_metric = None

    def reward(value):
        if getattr(metric_function, 'comp_func', min) == max:
            return float(value)
        return 1.0 / (float(value) + eps)

    for iteration in range(num_iterations):
        for ant in range(num_ants):
            probabilities = pheromone ** alpha
            probabilities = probabilities / probabilities.sum()
            selected_nodes = rng.choice(
                points_pool,
                size=num_points,
                replace=False,
                p=probabilities,
            ).tolist()

            times, _ = state_function(env=env, points=selected_nodes, **kwargs)
            metric_value = metric_function(times, **kwargs)

            if best_metric is None or metric_function.compare(best_metric, metric_value) == metric_value:
                best_metric = metric_value
                best_solution = list(selected_nodes)

            pheromone_delta = q * reward(metric_value)
            for node in selected_nodes:
                pheromone[node_to_index[node]] += pheromone_delta

        pheromone *= (1.0 - evaporation_rate)
        pheromone = np.clip(pheromone, eps, None)


    return best_solution, best_metric, pheromone


# Запуск ACO для одной точки размещения.
best_nodes_aco, best_metric_aco, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=1,
    num_ants=30,
    num_iterations=20,
    alpha=2.0,
    evaporation_rate=0.15,
    q=1.0,
    random_state=42,
)

print('ACO Лучшее решение:', best_nodes_aco)
print('ACO Метрика:', best_metric_aco)

aco_times, _ = state(G, points=best_nodes_aco)
print('Среднее время прибытия ACO:', ArrivalTime()(aco_times))

In [ ]:
# Оценка результата
times, _ = state(G, points=best_nodes_aco)
g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
g_nodes['arrival_time'] = times
psch = {node: f'ПСЧ {i+1}' for i, node in enumerate(best_nodes_aco)}
print_results(g_nodes, psch, figsize=(10,10))

In [ ]:
# Дополнительный пример: поиск нескольких точек размещения.

best_solution_multi, best_metric_multi, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=4,
    num_ants=30,
    num_iterations=20,
    alpha=2.0,
    evaporation_rate=0.2,
    q=2.0,
    random_state=2025,
)

print('ACO Лучшие 4 точки:', best_solution_multi)
print('ACO Метрика для 4 точек:', best_metric_multi)

In [ ]:
# Оценка результата
times, _ = state(G, points=best_solution_multi)
g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
g_nodes['arrival_time'] = times
psch = {node: f'ПСЧ {i+1}' for i, node in enumerate(best_solution_multi)}
print_results(g_nodes, psch, figsize=(10,10))

In [ ]:
# Дополнительно: Оценка расположения феромона
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = pheromone
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10))
g_nodes.plot(ax=ax1, column='pheromone', legend=True)
g_nodes.plot(ax=ax2, column='pheromone', legend=True)
g_nodes.loc[best_solution_multi].plot(ax=ax2, color='red')

plt.show()

In [ ]:
# Дополнительный пример: Оценка расположения феромона

best_solution_multi, best_metric_multi, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=3,
    num_ants=50,
    num_iterations=20,
    alpha=1.0,
    evaporation_rate=0.10,
    q=1.0,
    random_state=2025,
)

# Печать карты феромонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = pheromone
g_nodes.plot(column='pheromone', legend=True)
g_nodes[best_solution_multi]

In [ ]:
g_nodes.loc[best_solution_multi]

### Доработка метода разброса феромона

Из предыдущих экспериментов видно, что разброс феромона точечно не дает эффективного результата, т.к. большое количество узлов приводит к тому, что даже при высоком количестве феромона в одном узле, вероятность его выбора невысока, к тому же это не дает возможности выбора соседних узлов. Поэтому предлагается использовать разброс феромона не на конкретный узел в котором побывал муравей (выбран в качестве места выбора дислокации одного из подразделений пожарной охраны) а все в некотором окружении.

$$
P_i = q * ((L_{max}-L_i) / L_{max})
$$

Проверка размещения феромона:

In [ ]:
node_id = 3010

times = nx.single_source_dijkstra_path_length(G, node_id, cutoff=1, weight='travel_time')
times

In [ ]:
def pheromone_get_by_distance(l, l_max=1, q=1):
    return q * (l_max - l) / l_max

pheromone_get_by_distance(0.4)

In [ ]:
times_c = {k:pheromone_get_by_distance(t) for k,t in times.items()}
times_c

In [ ]:
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = 0.0
g_nodes.loc[list(times_c.keys()), 'pheromone'] = list(times_c.values())
g_nodes['pheromone']

In [ ]:
g_nodes.plot(column='pheromone')

In [ ]:
# все вместе
times = nx.single_source_dijkstra_path_length(G, node_id, cutoff=1, weight='travel_time')
times_c = {k:pheromone_get_by_distance(t) for k,t in times.items()}

g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = 0.0
g_nodes.loc[list(times_c.keys()), 'pheromone'] = list(times_c.values())


### Доработка исходной реализации с разбросом феромона

In [ ]:
from genesis.states import FirstArrivalUnitState
from genesis.metrics import ArrivalTime, CoverIndex
from genesis.best_points import BestNodeHillClimbing
from genesis.utils import timing

state = FirstArrivalUnitState()
metric = ArrivalTime()

def aco_best_points(env,
                    state_function,
                    metric_function,
                    num_points=1,
                    num_ants=20,
                    num_iterations=30,
                    alpha=1.0,
                    beta=1.0,
                    evaporation_rate=0.1,
                    q=1.0,
                    l_max=1.0,
                    points_pool=None,
                    random_state=None,
                    **kwargs):
    """Ищет лучшие узлы графа методом муравьиной колонии.

    Args:
        env (nx.Graph): граф.
        state_function (StateBase): вычисление времен прибытия от множества точек.
        metric_function (MetricBase): метрика для оценки качества размещения.
        num_points (int): число точек размещения.
        num_ants (int): число маршрутов (муравьев) за итерацию.
        num_iterations (int): число итераций.
        alpha (float): степень влияния феромона на выбор узлов.
        evaporation_rate (float): скорость испарения феромона.
        q (float): множитель для добавления феромона.
        l_max (float): максимальное время в пути для разброса феромона.
        points_pool (iterable): список узлов кандидатов.
        random_state (int): сид для генератора случайных чисел.
        **kwargs: дополнительные параметры для state_function (например area).

    Returns:
        tuple: (best_nodes, best_metric, pheromone)
    """

    if points_pool is None:
        points_pool = list(env.nodes())
    else:
        points_pool = list(points_pool)

    if len(points_pool) == 0:
        raise ValueError('points_pool не может быть пустым')

    num_points = min(num_points, len(points_pool))
    node_to_index = {node: i for i, node in enumerate(points_pool)}
    pheromone = np.ones(len(points_pool), dtype=float)
    # pheromone = 0.1 * pheromone
    rng = np.random.default_rng(random_state)
    eps = 1e-8

    best_solution = None
    best_metric = None
    worst_metric = None

    def reward(value):
        if getattr(metric_function, 'comp_func', min) == max:
            return float(value)
        return q * worst_metric / (float(value) + eps)
        # return q / (float(value) + eps)
        # return 1.0 / (float(value) + eps)
        # return float(cur_value - value)

    def pheromone_get_by_distance(l):   #, l_max=1, q=1):
        return (l_max - l) / l_max

    # # Определение стартовой метрики от которой будем отталкиваться
    # probabilities = pheromone ** alpha
    # probabilities = probabilities / probabilities.sum()
    # selected_nodes = rng.choice(
    #             points_pool,
    #             size=num_points,
    #             replace=False,
    #             p=probabilities,
    #         ).tolist()
    # times, _ = state_function(env=env, points=selected_nodes, **kwargs)
    # metric_on_iter = metric_function(times, **kwargs)

    for iteration in range(num_iterations):
        print(f'Итерация {iteration}: ', end='')
        # best_nodes = 0
        sn = []
        for ant in range(num_ants):
            print('.', end='')
            probabilities = pheromone ** alpha
            probabilities = probabilities / probabilities.sum()
            selected_nodes = rng.choice(
                points_pool,
                size=num_points,
                replace=False,
                p=probabilities,
            ).tolist()
            sn.extend(selected_nodes)

            times, _ = state_function(env=env, points=selected_nodes, **kwargs)
            metric_value = metric_function(times, **kwargs)

            if best_metric is None or metric_function.compare(best_metric, metric_value) == metric_value:
                # best_nodes += 1
                best_metric = metric_value
                best_solution = list(selected_nodes)

            if worst_metric is None or metric_function.compare(worst_metric, metric_value) == worst_metric:
                worst_metric = metric_value
                # print('worst_metric', worst_metric)
            
            reward_cur = reward(metric_value) ** beta
            # print(metric_value, reward_cur)

            # if metric_on_iter is None or metric_function.compare(metric_on_iter, metric_value) == metric_value:
                # pheromone_delta = q * reward(metric_value)
            times = nx.multi_source_dijkstra_path_length(env, selected_nodes, cutoff=l_max, weight='travel_time')
            times = {k:pheromone_get_by_distance(t) for k,t in times.items()}
            # pheromone.loc[list(times.keys()), 'pheromone'] = list(times.values())
            idx, vals = zip(*times.items())  # два кортежа
            # pheromone[np.array(idx)] = np.array(vals)
            for node, pheromone_delta in times.items():
                # pheromone[node_to_index[node]] += q * pheromone_delta * reward(metric_value, metric_on_iter)
                # print(
                #     pheromone[node_to_index[node]],
                #     q * pheromone_delta * reward(metric_value)
                # )
                # pheromone[node_to_index[node]] += q * pheromone_delta * reward(metric_value)
                # pheromone[node_to_index[node]] += pheromone_delta * reward(metric_value) ** beta
                # pheromone[node_to_index[node]] = max(
                #     pheromone[node_to_index[node]],
                #     pheromone_delta * reward(metric_value) ** beta
                # )
                pheromone[node_to_index[node]] = max(
                    pheromone[node_to_index[node]],
                    pheromone_delta * reward_cur,
                    # reward_cur
                )                

        print('->', best_metric)


        # g_nodes = ox.graph_to_gdfs(env, edges=False)
        # fig, (ax1, ax2) = plt.subplots(1, 2)   #, figsize=(15, 10))
        # g_nodes['pheromone'] = pheromone
        # g_nodes.plot(ax=ax1, column='pheromone')   #, legend=True)
        # g_nodes.loc[sn].plot(ax=ax1, color='white')        

        pheromone *= (1.0 - evaporation_rate)
        pheromone = np.clip(pheromone, eps, None) # Вот здесь понять, что это
        
        # g_nodes['pheromone'] = pheromone
        # g_nodes.plot(ax=ax2, column='pheromone')   #, legend=True)
        # g_nodes.loc[best_solution].plot(ax=ax2, color='red')
        # plt.show()

        # metric_on_iter = best_metric


    return best_solution, best_metric, pheromone


# Запуск ACO для одной точки размещения.
best_nodes_aco, best_metric_aco, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=1,
    num_ants=20,
    num_iterations=30,
    alpha=3.0,
    beta=1.0,
    evaporation_rate=0.2,
    q=10.0,
    l_max=4,
    random_state=29,
)

print('ACO Лучшее решение:', best_nodes_aco)
print('ACO Метрика:', best_metric_aco)

aco_times, _ = state(G, points=best_nodes_aco)
print('Среднее время прибытия ACO:', ArrivalTime()(aco_times))

In [ ]:
# Запуск ACO для одной точки размещения.
best_nodes_aco, best_metric_aco, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=4,
    num_ants=30,
    num_iterations=30,
    alpha=2.0,
    beta=1.0,
    evaporation_rate=0.05,
    q=1.0,
    l_max=4,
    random_state=41,
)

print('ACO Лучшее решение:', best_nodes_aco)
print('ACO Метрика:', best_metric_aco)

aco_times, _ = state(G, points=best_nodes_aco)
print('Среднее время прибытия ACO:', ArrivalTime()(aco_times))

In [ ]:
# Печать карты феромонов

g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = pheromone
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10))
g_nodes.plot(ax=ax1, column='pheromone', legend=True)
g_nodes.plot(ax=ax2, column='pheromone', legend=True)
g_nodes.loc[best_nodes_aco].plot(ax=ax2, color='red')

plt.show()

In [ ]:
best_solution_multi, best_metric_multi, pheromone = timing(aco_best_points)(
    env=G,
    state_function=state,
    metric_function=metric,
    num_points=4,
    num_ants=30,
    num_iterations=5,
    alpha=1.0,
    evaporation_rate=0.15,
    q=1.0,
    l_max=2,
    random_state=34,
)

# Печать карты феромонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes['pheromone'] = pheromone
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10))
g_nodes.plot(ax=ax1, column='pheromone', legend=True)
g_nodes.plot(ax=ax2, column='pheromone', legend=True)
g_nodes.loc[best_solution_multi].plot(ax=ax2, color='red')

plt.show()

## Реализация в соответсвии с genesis

In [ ]:
# Проверка для новой реализации
from genesis.stop_cases import ItersStopCase
from genesis.point_selectors import RandomNodesSelector


aco_mclp_solution = ACO_MCLP(
    state_function     = state,
    metric_function    = metric,
    point_selector     = None,
    # stop_case_function = ItersStopCase,
    points_pool        = G.nodes(),
)

best_solution_multi, best_metric_multi, _ = aco_mclp_solution(
    env=G,
)

print('ACO Лучшие 3 точки:', best_solution_multi)
print('ACO Метрика для 3 точек:', best_metric_multi)